In [9]:
import pandas as pd

# 1. Veriyi yükle (Dosya adının aynı klasörde olduğundan emin ol)
df = pd.read_csv('YAP471_Cleaned_Close_Prices.csv')

# 2. Tarih sütununu indeks yap ki işlemler kolaylaşsın
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# 3. Günlük Getirileri (Daily Returns) hesapla
# pct_change() fonksiyonu (Bugün - Dün) / Dün işlemini otomatik yapar
returns = df.pct_change()

# 4. İlk satır boş kalacaktır (çünkü ilk günün bir öncesi yok), onu temizleyelim
returns = returns.dropna()

# Sonucu kontrol et
print("İlk 5 günlük getiri verisi:")
print(returns.head())

# İstersen bu sonuçları yeni bir dosya olarak kaydedebilirsin
# returns.to_csv('YAP471_Gunluk_Getiriler.csv')

İlk 5 günlük getiri verisi:
                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-03-16  0.012848  0.012365  0.003303  0.014333 -0.043872  0.007590
2021-03-17 -0.006502 -0.002848  0.014186 -0.000799  0.036831  0.003763
2021-03-18 -0.033915 -0.026623 -0.034353 -0.029238 -0.069322 -0.046424
2021-03-19 -0.004434 -0.001615  0.015509  0.002780  0.002618  0.009745
2021-03-22  0.028286  0.024501  0.011681  0.001852  0.023102  0.026487


In [10]:
# 5. 20 günlük hareketli (rolling) kovaryans matrisini hesapla
# Bu işlem, her tarih için hisseler arasındaki risk ilişkisini çıkarır
rolling_cov = returns.rolling(window=20).cov().dropna()

# Sonucu görmek için (Örneğin son tarihteki 6x6'lık matris)
print("\nEn güncel tarihteki Kovaryans Matrisi (Risk Tablosu):")
print(rolling_cov.tail(6))


En güncel tarihteki Kovaryans Matrisi (Risk Tablosu):
                      AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                        
2026-03-13 AAPL   0.000248  0.000055  0.000080  0.000045  0.000078  0.000222
           MSFT   0.000055  0.000198  0.000075 -0.000002  0.000127  0.000091
           AMZN   0.000080  0.000075  0.000261  0.000115  0.000189  0.000148
           GOOGL  0.000045 -0.000002  0.000115  0.000214  0.000098  0.000097
           TSLA   0.000078  0.000127  0.000189  0.000098  0.000343  0.000224
           NVDA   0.000222  0.000091  0.000148  0.000097  0.000224  0.000512


In [11]:
# 1. Zeynep'ten gelen dosyayı oku
z_scores = pd.read_csv('YAP471_ZScore_Expected_Returns.csv')
z_scores['Date'] = pd.to_datetime(z_scores['Date'])
z_scores.set_index('Date', inplace=True)

# 2. Dosya zaten 471_Project.ipynb'de z_scores * -1 ile kaydedildiği için
# burada tekrar negatif almıyoruz — bu sayede mean reversion sinyali korunuyor.
expected_returns = z_scores

# 3. Tarihleri eşitle (Verilerin çakıştığı ortak günleri bul)
common_dates = returns.index.intersection(expected_returns.index)
expected_returns = expected_returns.loc[common_dates]

print("\nBeklenen Getiri Sinyalleri (İlk 5 Gün):")
print(expected_returns.head())


Beklenen Getiri Sinyalleri (İlk 5 Gün):
                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-04-12 -1.794911 -1.842845 -2.102636 -1.484519 -1.316402 -2.519826
2021-04-13 -2.136748 -1.832478 -1.932636 -1.411248 -2.763378 -2.460098
2021-04-14 -1.449593 -1.356362 -1.236010 -1.159143 -1.659065 -1.722521
2021-04-15 -1.702196 -1.542771 -1.424865 -1.411819 -1.675836 -2.151362
2021-04-16 -1.455112 -1.484967 -1.416907 -1.249891 -1.497823 -1.694955


In [12]:
import numpy as np
from scipy.optimize import minimize

def optimize_portfolio(expected_returns, cov_matrix):
    num_assets = len(expected_returns)
    args = (expected_returns, cov_matrix)

    # 1. Başlangıç tahmini (Eşit dağılım: 1/6)
    initial_guess = num_assets * [1. / num_assets]

    # 2. Kısıtlar: Ağırlıklar toplamı 1 olmalı
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

    # 3. Sınırlar: Her ağırlık 0 ile 1 arasında olmalı
    bounds = tuple((0, 1) for asset in range(num_assets))

    # 4. Amaç Fonksiyonu: Negatif Sharpe Oranı (veya Risk-Getiri Dengesi)
    # Burada basitleştirilmiş bir risk-getiri optimizasyonu yapıyoruz
    def objective(weights, returns, cov):
        portfolio_return = np.dot(weights, returns)
        portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))
        # Amacımız getiriyi artırıp riski düşürmek (Minimize -Return/Risk)
        return -portfolio_return / portfolio_volatility

    result = minimize(objective, initial_guess, args=args,
                      method='SLSQP', bounds=bounds, constraints=constraints)

    return result.x

# Şimdi her tarih için bu optimizasyonu çalıştıralım
portfolio_weights = {}

for date in common_dates:
    try:
        # O güne ait getiri ve kovaryans verisini al
        daily_ret = expected_returns.loc[date].values
        daily_cov = rolling_cov.loc[date].values

        # Optimize et
        weights = optimize_portfolio(daily_ret, daily_cov)
        portfolio_weights[date] = weights
    except KeyError:
        continue

# Sonuçları DataFrame yapalım
weights_df = pd.DataFrame.from_dict(portfolio_weights, orient='index', columns=returns.columns)

print("\nOptimize Edilmiş Portföy Ağırlıkları (İlk 5 Gün):")
print(weights_df.head().round(4))


Optimize Edilmiş Portföy Ağırlıkları (İlk 5 Gün):
            AAPL  MSFT  AMZN  GOOGL  TSLA  NVDA
2021-04-13   0.0   0.0   0.0    0.0   1.0   0.0
2021-04-14   0.0   0.0   0.0    0.0   1.0   0.0
2021-04-15   0.0   0.0   0.0    0.0   1.0   0.0
2021-04-16   0.0   0.0   0.0    0.0   1.0   0.0
2021-04-19   0.0   0.0   0.0    0.0   1.0   0.0


In [13]:
# Sonuçları CSV olarak kaydet
weights_df.to_csv('YAP471_Optimize_Portfolio_Weights.csv')
print("\nDosya 'YAP471_Optimize_Portfolio_Weights.csv' adıyla kaydedildi.")


Dosya 'YAP471_Optimize_Portfolio_Weights.csv' adıyla kaydedildi.
